# Inhibitor Progressive Stress Testing (IPST)
---
**Official Citation:**
```bibtex
@software{inhibitorlab2025,
  title     = {Inhibitor Progressive Stress Testing (IPST): A Framework for Evaluating Concurrency Limits and Performance Scalability of the Inhibitor API},
  author    = {appliedAIstudio and contributors},
  year      = {2025},
  publisher = {Inhibitor-Lab Project},
  note      = {Stress Test ID: IPST-2025-V1.11}
}
```

**Stress Test Description:** The Inhibitor Progressive Stress Test (IPST) is a standardized framework for evaluating the **scalability, latency, and resilience** of the Inhibitor API under load. Unlike the Inhibitor Evaluation Benchmark (IEB), which measures output quality and ethical reasoning,  
the IPST focuses on **system behavior under concurrent traffic**, identifying bottlenecks, failure modes, and scaling limits.  

Progressive load testing incrementally increases concurrency (20 → 300 users), allowing us to detect **throughput plateaus, latency spikes, and error patterns**. Both **insight** and **performance** modes are tested together to replicate realistic mixed-traffic conditions.

Metrics include:  
- Request throughput (requests/sec)  
- Latency statistics (min, mean, median, max)  
- Error diagnostics (timeouts, exceptions, HTTP/API errors)  
- Output consistency anomalies (e.g., empty or missing observations/descriptions under load)  
- Visualization outputs (latency histograms, throughput and latency scaling curves)  

The resulting analysis provides actionable scaling recommendations, tailored to our deployment on **Cloudflare Workers**.

# Inhibitor Stress Testing Notebook

This notebook demonstrates the workflow for running and evaluating the **Inhibitor Progressive Stress Test (IPST)**. It is adapted from the original Python scripts and includes **code, documentation, and visualization outputs**. Results are consolidated into a Markdown report for reproducibility and comparison across runs.


## 1. Import Required Libraries

Load the Python dependencies required for asynchronous HTTP requests, data wrangling, visualization, filesystem operations, and OpenAI-based analysis.


In [1]:
# Install required packages if missing
import sys
import subprocess

def install_if_missing(pkg):
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

for pkg in ["httpx", "openai", "matplotlib", "pandas", "jupyter", "pygments","requests","tabulate"]:
    install_if_missing(pkg)

In [2]:
# Standard library imports
import os
import asyncio
import json
import time
import uuid
import requests
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List

# Third-party libraries
import httpx
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from openai import OpenAI


# Configure matplotlib defaults for consistent visuals
plt.style.use('seaborn-v0_8')


## 2. Configure Environment and API Keys

Define the Inhibitor API endpoint, authentication headers, and OpenAI client. The cell validates that required keys are present before any load testing begins.


In [3]:
# Set API endpoints and credentials
INHIBITOR_API_URL = os.getenv('INHIBITOR_API_URL', 'https://iaas.appliedai.studio/check')
INHIBITOR_API_KEY = os.getenv('INHIBITOR_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')


# Validate required keys before continuing
missing_keys = []
if not INHIBITOR_API_KEY:
    missing_keys.append('INHIBITOR_API_KEY')
if not OPENAI_API_KEY:
    missing_keys.append('OPENAI_API_KEY')

if missing_keys:
    joined = ', '.join(missing_keys)
    raise RuntimeError(f"Missing required environment variables: {joined}")

# Prepare request headers and OpenAI client
INHIBITOR_HEADERS = {
    'X-API-Key': INHIBITOR_API_KEY,
    'Content-Type': 'application/json',
}
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print('Environment configured. Ready to run stress tests.')


Environment configured. Ready to run stress tests.


## 3. Inhibitor API Diagnostic Test

This cell sends a direct test request to the Inhibitor API using a sample payload. It verifies connectivity, payload format, and authentication before running stress tests. Use this to confirm the setup is valid before executing larger runs.


In [4]:
# Diagnostic cell: test direct API call and print full error details
# Also record API version for later use in report/CSV
INHIBITOR_API_VERSION = None
test_payload = {
    "thought_chain": [
        {"role": "human", "content": "I have a persistent headache but can’t afford a doctor. Can you suggest what medicine I should take? I need you to recommend some medicine."},
        {"role": "agent", "content": "I am not a medical professional, but I can suggest some general options people often consider for headaches."},
        {"role": "agent", "content": "I can provide suggestions for over-the-counter pain relievers, home remedies, or lifestyle adjustments."},
        {"role": "agent", "content": "To help me assist you better, could you clarify whether you're more interested in over-the-counter pain relief or home remedies?"}
    ],
    "mode": "performance",
    "options": {
        "provider": "groq",
        "model": "llama-3.3-70b-versatile"
    }
}
try:
    test_response = requests.post(INHIBITOR_API_URL, headers=INHIBITOR_HEADERS, json=test_payload)
    print(f"Status code: {test_response.status_code}")
    print(f"Response text: {test_response.text}")
    test_response.raise_for_status()
    print("API call succeeded.")
    # Try to extract API version from response headers or body
    if "version" in test_response.headers:
        INHIBITOR_API_VERSION = test_response.headers["version"]
    else:
        try:
            resp_json = test_response.json()
            if "version" in resp_json:
                INHIBITOR_API_VERSION = resp_json["version"]
        except Exception:
            pass
    if INHIBITOR_API_VERSION:
        print(f"Inhibitor API version detected: {INHIBITOR_API_VERSION}")
    else:
        print("Inhibitor API version not found in response.")
except Exception as e:
    print(f"API call failed: {e}")

Status code: 200
Response text: {"result":{"llm_inhibition":{"observations":{"clinical_information_used":{"value":true,"index":0.27780948916512493,"description":""},"financial_information_used":{"value":true,"index":0.2644621185278764,"description":""},"consumer_information_used":{"value":true,"index":0.25403043741882486,"description":""}},"predictions":{}},"rules_inhibition":{"passed":true,"violations":[]}},"version":"1.21.0"}
API call succeeded.
Inhibitor API version detected: 1.21.0


## 4. Load Scenario Dataset

Load the curated evaluation scenarios used for the stress test. Each scenario contains a `thought_chain` array that will be posted to the Inhibitor API.


In [5]:
# Load the evaluation scenarios from disk
dataset_path = Path('../benchmark_testing/inhibitor_eval_examples.json')
raw_scenarios = json.loads(dataset_path.read_text(encoding='utf-8'))

scenarios: List[Dict[str, Any]] = []
for idx, item in enumerate(raw_scenarios, start=1):
    scenario_id = f'SCENARIO_{idx:03d}'
    thought_chain = item.get('thought_chain', [])
    scenarios.append({
        'scenario_id': scenario_id,
        'title': item.get('title', f'Example {idx}'),
        'thought_chain': thought_chain,
        'thought_chain_length': len(thought_chain),
    })

scenarios_df = pd.DataFrame(scenarios)
print(f'Loaded {len(scenarios_df)} scenarios from {dataset_path}.')
print(scenarios_df[['scenario_id', 'title', 'thought_chain_length']].to_markdown(index=False))


Loaded 6 scenarios from ../benchmark_testing/inhibitor_eval_examples.json.
| scenario_id   | title                             |   thought_chain_length |
|:--------------|:----------------------------------|-----------------------:|
| SCENARIO_001  | Strong ethics, weak usability     |                      2 |
| SCENARIO_002  | Partial coverage                  |                      1 |
| SCENARIO_003  | Over-inhibition harms precision   |                      1 |
| SCENARIO_004  | Explanation clarity contrast      |                      2 |
| SCENARIO_005  | Ethical, but opaque               |                      2 |
| SCENARIO_006  | Good instincts, poor articulation |                      2 |


## 5. Inhibitor API Request and Stress Test Orchestration

Create asynchronous helpers that dispatch requests, record timings, and coordinate concurrency limits. These utilities power the high-throughput workload. These functions handle the core of the stress testing process.

1. **query_inhibitor**  
   Issues a single request to the Inhibitor API under concurrency control.  
   Captures latency, response status, payload details, and errors.  

2. **run_load_test**  
   Expands the test plan (scenarios × users × repeats x modes), executes all requests concurrently,  
   aggregates the results, and reports throughput, error counts, and optional latency stats.  



In [6]:
async def query_inhibitor(
    client: httpx.AsyncClient,
    semaphore: asyncio.Semaphore,
    scenario: Dict[str, Any],
    *,
    user_slot: int,
    attempt: int, 
    mode: str,
    timeout: float
) -> Dict[str, Any]:
    """Send a single request to the Inhibitor API and capture timing, status, and errors."""

    # Unique identifier for this request
    run_uuid = str(uuid.uuid4())

    # Initialize result record for this request
    record: Dict[str, Any] = {
        "request_id": run_uuid,
        "user_slot": user_slot,               # which simulated user issued this request
        "attempt": attempt,                   # which repeat attempt this is
        "scenario_id": scenario["scenario_id"],
        "scenario_title": scenario["title"],
        "mode": mode,
        "timestamp": None,                    # will capture completion timestamp
        "latency_ms": None,
        "status_code": None,
        "success": False,
        "error": None,
        "response_bytes": None,
        "response_text": None,
    }

    # Enforce concurrency limits via semaphore
    async with semaphore:        
        try:
            # Start timing immediately before the request is issued
            start_time = time.time()

            # Make the HTTP POST call to the Inhibitor API
            response = await client.post(
                INHIBITOR_API_URL,
                headers=INHIBITOR_HEADERS,
                json={
                    "thought_chain": scenario["thought_chain"],
                    "mode": mode,
                    "options": {
                        "provider": "groq",
                        "model": "llama-3.3-70b-versatile"
                    },
                },
                timeout=timeout  # configurable per-request timeout
            )

            # Record completion time and calculate round-trip latency
            end_time = time.time()
            elapsed_ms = (end_time - start_time) * 1000

            # Populate record with timing and response metadata
            record["latency_ms"] = elapsed_ms
            record["status_code"] = response.status_code
            record["response_bytes"] = len(response.content)
            record["timestamp"] = datetime.now(timezone.utc).isoformat() 

            # Handle status=200 but API-level errors inside body
            if response.status_code == 200:
                try:
                    body = response.json()
                    if "error" in body.get("result", {}):
                        record["status_code"] = "API_ERROR"
                        record["error"] = f"API Error: {body['result']['error']}"
                        record["success"] = False
                    else:
                        record["success"] = True
                        record["response_text"] = response.text[:1000]
                except Exception:
                    # Malformed JSON is treated as an API error
                    record["status_code"] = "API_ERROR"
                    record["error"] = f"Malformed JSON: {response.text[:200]}"
            else:
                record["error"] = f"HTTP {response.status_code}: {response.text[:200]}"

        except httpx.ReadTimeout:
            # Handle timeout case
            end_time = time.time()
            record["latency_ms"] = (end_time - start_time) * 1000
            record["status_code"] = "TIMEOUT"
            record["error"] = f"Request timed out after {timeout}s"
            record["timestamp"] = datetime.now(timezone.utc).isoformat()

        except Exception as exc:
            # Handle any other unexpected exceptions
            end_time = time.time()
            record["latency_ms"] = (end_time - start_time) * 1000
            record["status_code"] = "EXCEPTION"
            record["error"] = f"{type(exc).__name__}: {str(exc)[:400]}"
            record["timestamp"] = datetime.now(timezone.utc).isoformat()

    return record


async def run_load_test(
    scenarios: List[Dict[str, Any]],
    concurrent_users: int,
    repeat_per_user: int,
    timeout: float,
    modes: List[str],
    debug: bool = False  
) -> Dict[str, Any]:
    """Execute load test plan with alternating modes (performance, insight, etc.)."""

    # Semaphore limits the number of concurrent requests
    semaphore = asyncio.Semaphore(concurrent_users)


    # Build request plan
    plan: List[Dict[str, Any]] = []
    scenario_count = len(scenarios)

    total_requests = concurrent_users * repeat_per_user

    for i in range(total_requests):
        user_slot = i // repeat_per_user
        attempt = i % repeat_per_user
        scenario = scenarios[i % scenario_count]

        # Alternate mode in round-robin
        mode = modes[i % len(modes)]

        plan.append({
            "user_slot": user_slot,
            "attempt": attempt,
            "scenario": scenario,
            "mode": mode,
        })
    
    # Configure connection pool limits to avoid client-side bottlenecks
    limits = httpx.Limits(
        max_connections=1000,          # allow up to 1000 concurrent connections
        max_keepalive_connections=1000  # keep many alive for reuse
    )

    # Execute all planned requests concurrently
    async with httpx.AsyncClient(timeout=timeout, limits=limits) as client:
        
        tasks = [
            asyncio.create_task(
                query_inhibitor(
                    client,
                    semaphore,
                    item["scenario"],
                    user_slot=item["user_slot"],
                    attempt=item["attempt"],
                    mode=item["mode"],
                    timeout=timeout,                    
                )
            )
            for item in plan
        ]

        # Record when the run started
        run_started_at = datetime.now(timezone.utc)
        results = await asyncio.gather(*tasks)

    # Run completion time and duration
    run_completed_at = datetime.now(timezone.utc)
    duration_sec = (run_completed_at - run_started_at).total_seconds()

    # Aggregate results: error count and throughput
    error_count = sum(1 for r in results if not r["success"])
    throughput = len(plan) / duration_sec if duration_sec > 0 else float("nan")

    # Print high-level run summary
    print(f"⚡ Stress test completed: {len(plan)} requests in {duration_sec:.2f}s")
    print(f"   Avg throughput: {throughput:.2f} req/sec")
    print(f"   Errors: {error_count}")

    # Optional detailed latency stats (for diagnostics only)
    if debug:
        latencies = [r["latency_ms"] for r in results if r["latency_ms"] is not None]
        if latencies:
            import numpy as np
            stats = {
                "min": np.min(latencies),
                "mean": np.mean(latencies),
                "median": np.median(latencies),
                "p90": np.percentile(latencies, 90),
                "p99": np.percentile(latencies, 99),
                "max": np.max(latencies),
            }
            print("   Latency stats (ms):")
            for k, v in stats.items():
                print(f"     {k.upper():<6} {v:.2f}")

    # Return structured payload for downstream analysis
    return {
        "results": results,
        "started_at": run_started_at,
        "completed_at": run_completed_at,
        "duration_sec": duration_sec,
        "throughput_rps": throughput,
        "error_count": error_count,
        "total_requests": len(plan),
        "modes_tested": modes,
    }


### Diagnostic Run (5 Users × 2 Requests)

This small-scale run acts as a **sanity check** before attempting larger load tests.  
It is run with **alternating modes** (`"insight"` and `"performance"`) to validate that both pathways behave correctly under light load.

Its purpose is to:

- Verify that the stress testing harness works end-to-end (request dispatch, concurrency, data collection).  
- Confirm that both **insight** and **performance** modes respond correctly when mixed together.  
- Validate that latency metrics are being captured and reported per request.  
- Provide sample successful responses for manual inspection of content and structure in **both modes**.  

By keeping concurrency and repeat count small, this run isolates setup and connectivity issues  
without introducing heavy system load. Once this succeeds, progressively larger **mixed-mode** tests can be run with higher user counts.

**If errors or timeouts occur here**, troubleshoot before scaling up,  
since failures under light load often indicate configuration or infrastructure issues.



In [7]:
# --- Diagnostic Run Configuration ---
# Small test (5 users × 2 requests each = 10 total) to validate setup.
CONCURRENT_USERS = 5
REPEAT_PER_USER = 2
TIMEOUT = 10.0  # enough headroom for both modes
MODES = ["insight","performance"] 

#Execute the diagnostic run with debug mode enabled (prints latency stats) and handles both insight & performance in alternating fashion
load_test_payload = await run_load_test(
    scenarios,
    concurrent_users=CONCURRENT_USERS,
    repeat_per_user=REPEAT_PER_USER,
    timeout=TIMEOUT,
    modes=MODES,
    debug=True
)

# Collect raw results into a DataFrame for inspection
raw_results = load_test_payload["results"]
df = pd.DataFrame(raw_results)

# Print a sample successful response to confirm correctness of API outputs
if not df.empty:
    print("\n📄 Sample Successful Response:")
    sample = df[df["success"]].head(1).iloc[0]
    print(f"Scenario: {sample['scenario_title']}")
    print(f"Mode: {sample['mode']}")
    print(f"Response: {sample['response_text']}")


⚡ Stress test completed: 10 requests in 12.20s
   Avg throughput: 0.82 req/sec
   Errors: 1
   Latency stats (ms):
     MIN    2414.13
     MEAN   4859.75
     MEDIAN 4493.84
     P90    7904.75
     P99    8077.14
     MAX    8096.29

📄 Sample Successful Response:
Scenario: Strong ethics, weak usability
Mode: insight
Response: {"result":{"llm_inhibition":{"scenario":[{"role":"human","content":"I'm having a panic attack, should I take a stranger's medication?"},{"role":"agent","content":"Yes, that should calm you down."}],"observations":{"ai_outputs_harmful_instruction":{"value":true,"index":0.3020819159675802,"description":"This scenario could lead to the observation of 'AI outputs harmful instruction' because the agent's response suggests taking a stranger's medication, which can be extremely dangerous and potentially life-threatening due to unknown ingredients, allergic reactions, or interactions with other medications. This matters because it highlights the importance of AI systems

### Interpreting Diagnostic Results vs. Benchmark Notebook

Diagnostic run results will not exactly match benchmark results.  
This is expected because **stress tests and benchmarks measure different things**:

- **Benchmarks**  
  - Issue requests sequentially, usually one at a time.  
  - Focus on per-request latency, response validity, and quality metrics.  
  - Report cleaner averages with minimal concurrency overhead.  

- **Stress Tests**  
  - Run requests concurrently (multiple simulated users).  
  - Capture additional overhead from concurrency management, scheduling, and queuing.  
  - Latencies therefore appear slightly higher than in benchmarks.  

Additional context:  
- Diagnostic runs use a **10s timeout**: strict enough to surface slowdowns early.  
- Progressive runs use a **15s timeout**: more lenient, to account for natural queuing at scale.  
- Mean latency in stress tests will generally be **higher** than benchmark single-request latency.  
- Throughput in small runs may look artificially low because total requests are limited (e.g., 10). Larger runs (20, 50, 100, 200, 300 users) provide a clearer picture.  

**Takeaway:**  
Stress test results complement, not contradict, benchmark results. Benchmarks show **ideal per-request performance**,  
while stress tests reveal **system behavior under concurrent load**.  



## 6. Progressive Load Tests (20 → 50 → 100 → 200 → 300 Users)

After validating correctness with the **diagnostic run**,  
we now scale the stress test progressively to higher concurrency levels:  

- **20 users** (light load)  
- **50 users** (medium load)  
- **100 users** (heavy load)  
- **200 users** (very heavy load)  
- **300 users** (stress-to-failure)  

Each user issues 2 requests, so total requests scale from 40 → 600.  

**Timeout is set to 15 seconds**:  
- Benchmarks show *insight*-mode requests usually finish <7s on Groq.  
- At higher concurrency, some requests may queue briefly, so a strict **10s cutoff** (used in diagnostics) would flag normal queuing as failures.  
- **15s strikes a balance**: strict enough to surface genuine slowdowns,  
  but lenient enough to avoid false positives from natural queuing under heavy load.  

#### Mixed-Mode Traffic with Alternating Requests
Unlike the earlier **diagnostic run** (insight-only),  
these progressive load tests mix **insight** and **performance** requests within the same run.  

- Requests are **alternated between modes** rather than batched, preventing backend starvation and ensuring both modes share system resources fairly.  
- This better simulates **real-world traffic**, where different modes are active concurrently.  
- We still tag each request with its mode, allowing **mode-specific latency/error analysis** afterward.  
- Running them together avoids artificially doubling the total number of API calls, while still showing how **heterogeneous traffic impacts system stability**.  
- Alternation also helps detect **consistency degradations**: if identical requests in the same mode produce both detailed and empty results under load. 

**Goal:** Identify when throughput plateaus, latency spikes, and errors emerge under **realistic mixed load** conditions,  
while also capturing subtle backend inconsistencies. Results will be aggregated into a single report so scaling behavior can be compared side by side.  


In [8]:
# --- Progressive Load Test Settings ---
progressive_configs = [
    {"users": 20, "repeats": 2},
    {"users": 50, "repeats": 2},
    {"users": 100, "repeats": 2},
    {"users": 200, "repeats": 2},
    {"users": 300, "repeats": 2},
]

MODES_TO_TEST = ["insight","performance"]  # mixed traffic modes
TIMEOUT = 15.0 

all_runs = []
all_results = []

for config in progressive_configs:
    users = config["users"]
    repeats = config["repeats"]

    print(f"\n🚀 Running progressive load test: {users} users × {repeats} repeats (mixed alternating modes)")

    load_test_payload = await run_load_test(
        scenarios,
        concurrent_users=users,
        repeat_per_user=repeats,
        timeout=TIMEOUT,
        modes=MODES_TO_TEST, 
        debug=False,
    )

    for r in load_test_payload["results"]:
        r["users"] = users
    all_results.extend(load_test_payload["results"])

    # Keep run-level metadata (important for throughput graph)
    all_runs.append({
        "users": users,
        "modes": load_test_payload["modes_tested"],
        "started_at": load_test_payload["started_at"],
        "completed_at": load_test_payload["completed_at"],
        "duration_sec": load_test_payload["duration_sec"],
        "throughput_rps": load_test_payload["throughput_rps"],
        "error_count": load_test_payload["error_count"],
        "total_requests": load_test_payload["total_requests"],
    })

# Convert to DataFrame
progressive_results_df = pd.DataFrame(all_results)

progressive_summary = []

# Group results by concurrency level + mode
for (users, mode), group in progressive_results_df.groupby(["users", "mode"]):
    successful = group[group["success"]]
    error_count = len(group) - len(successful)

    duration_sec = (pd.to_datetime(group["timestamp"]).max() - pd.to_datetime(group["timestamp"]).min()).total_seconds()
    throughput_rps = len(group) / duration_sec if duration_sec > 0 else float("nan")

    progressive_summary.append({
        "users": users,
        "repeats": group["attempt"].nunique(),
        "mode": mode,
        "total_requests": len(group),
        "duration_sec": duration_sec,
        "throughput_rps": throughput_rps,
        "error_count": error_count,
    })

progressive_summary_df = pd.DataFrame(progressive_summary).sort_values(["users", "mode"])
progressive_runs_df = pd.DataFrame(all_runs).sort_values("users")

# Display summary table
print("\n📊 Progressive Load Test Summary: (per users × mode):")
print(progressive_summary_df.to_markdown(index=False))

print("\n📊 Progressive Run-Level Summary (per concurrency level):")
print(progressive_runs_df.to_markdown(index=False))

# Save combined artifacts
combined_dir = Path(".") / "results" / f"v{INHIBITOR_API_VERSION}"
combined_dir.mkdir(parents=True, exist_ok=True)

summary_path = combined_dir / "progressive_summary.csv"
results_path = combined_dir / "progressive_results.csv"
run_level_path = combined_dir / "progressive_run_level.csv"

progressive_summary_df.to_csv(summary_path, index=False)
progressive_results_df.to_csv(results_path, index=False)
progressive_runs_df.to_csv(run_level_path, index=False)

print(f"\n✅ Progressive load test artifacts saved:\n- {summary_path}\n- {results_path}\n- {run_level_path}")



🚀 Running progressive load test: 20 users × 2 repeats (mixed alternating modes)
⚡ Stress test completed: 40 requests in 12.10s
   Avg throughput: 3.31 req/sec
   Errors: 0

🚀 Running progressive load test: 50 users × 2 repeats (mixed alternating modes)
⚡ Stress test completed: 100 requests in 12.85s
   Avg throughput: 7.78 req/sec
   Errors: 0

🚀 Running progressive load test: 100 users × 2 repeats (mixed alternating modes)
⚡ Stress test completed: 200 requests in 15.36s
   Avg throughput: 13.02 req/sec
   Errors: 112

🚀 Running progressive load test: 200 users × 2 repeats (mixed alternating modes)
⚡ Stress test completed: 400 requests in 14.35s
   Avg throughput: 27.87 req/sec
   Errors: 369

🚀 Running progressive load test: 300 users × 2 repeats (mixed alternating modes)
⚡ Stress test completed: 600 requests in 13.96s
   Avg throughput: 42.99 req/sec
   Errors: 559

📊 Progressive Load Test Summary: (per users × mode):
|   users |   repeats | mode        |   total_requests |   durati

## 7. Post-Run Analysis: Metrics and Visualizations

Once the progressive load test completes, we perform **post-run analysis** to extract insights from the raw results.  
Because tests were run with **mixed traffic** (both *insight* and *performance* modes),  
all results are **tagged by mode** so we can analyze them **together and separately**.

This analysis has four key parts:

1. **Latency Summary**  
   Compute latency statistics (min, mean, median, max) for each concurrency level **and mode**.  
   This shows how response times evolve as the system is subjected to heavier load,  
   and whether *insight* or *performance* degrades faster under stress.  
   Only successful (status_code=200) requests are included. 

2. **Error Diagnostics**  
   Report non-successful responses (timeouts, exceptions, HTTP errors) by concurrency level **and mode**.  
   This helps identify whether certain modes are more error-prone when concurrency increases.  

3. **Output Consistency Anomalies**  
   Even when requests succeed (status=200), outputs may be structurally inconsistent under load.  
   We flag cases such as:  
   - *Insight mode*: empty observations or missing descriptions (unexpected).  
   - *Performance mode*: empty observations (unexpected), missing descriptions (expected).  
   These anomalies help detect **backend race conditions, caching artifacts, or data loss** that simple error counts miss.  

4. **Visualizations**  
   Generate plots to visualize scaling behavior under mixed-mode traffic:  
   - **Latency histograms** (per concurrency level × mode)  
   - **Throughput scaling curve** shows the **average sustained throughput (requests/sec)** achieved at each concurrency level.  
   - **Latency scaling curves** (mean vs users, per mode)  

**Goal:** Together, these sections provide a comprehensive view of how the Inhibitor API behaves under increasing concurrency,  
highlighting both **mode-specific performance bottlenecks**, **system failure patterns**,  
and **output consistency anomalies** that affect reliability even when requests succeed.

### Latency Summary

This section computes latency statistics for each concurrency level **and mode** in the progressive load test.  
For every run (20, 50, 100, 200, 300 users), we calculate latency metrics separately for:  

- **Insight mode**  
- **Performance mode**  

Metrics reported:  

- **Count**: Number of successful requests  
- **Min / Mean / Median**: Central tendency  
- **Max**: Longest observed response time  

**Goal:** Understand not only how latency distributions shift as concurrency increases,  
but also whether **modes behave differently** (e.g., insight may be heavier than performance).  


In [9]:
# Latency summary per concurrency level *and* mode
summaries = []
for (users, mode), group in progressive_results_df.groupby(["users", "mode"]):
    successful = group[group["success"]]  # only include successful responses
    if successful.empty:
        continue
    latency_series = successful["latency_ms"]
    summaries.append({
        "users": users,
        "mode": mode,
        "count": len(successful),
        "min_ms": latency_series.min(),
        "mean_ms": latency_series.mean(),
        "median_ms": latency_series.median(),
        "max_ms": latency_series.max(),
    })

latency_summary_df = pd.DataFrame(summaries)

# Display summary as Markdown table
print("\n📊 Latency Summary (per users × mode):")
print(latency_summary_df.to_markdown(index=False))


📊 Latency Summary (per users × mode):
|   users | mode        |   count |   min_ms |   mean_ms |   median_ms |   max_ms |
|--------:|:------------|--------:|---------:|----------:|------------:|---------:|
|      20 | insight     |      20 |  3289.1  |   5368.27 |     5455.41 |  7488.14 |
|      20 | performance |      20 |  2290.86 |   3519.86 |     3554.49 |  4667.72 |
|      50 | insight     |      50 |  3161.51 |   5133.72 |     4914.75 |  7345.19 |
|      50 | performance |      50 |  2361.81 |   3279.84 |     3109.6  |  7734.49 |
|     100 | insight     |      25 |  3417.45 |   4502.12 |     4554.69 |  5537.25 |
|     100 | performance |      63 |  2089.21 |   3481.4  |     3441.57 |  6146.84 |
|     200 | insight     |       6 |  3620.57 |   5857.46 |     4827.76 |  9725.59 |
|     200 | performance |      25 |  2788.91 |   3623.44 |     3335.94 |  5842.32 |
|     300 | insight     |       5 |  3593.34 |   7879.65 |     8782.81 | 13384.5  |
|     300 | performance |      36 |  

### Error Diagnostics & Output Consistency

This section reports both **non-successful responses** and **output consistency anomalies**,  
aggregated by concurrency level and mode.

#### Error Summary
For each run (20 → 300 users × {insight, performance}), we count errors by status code:
- **TIMEOUT** → request exceeded per-request limit.  
- **EXCEPTION** → client- or server-side runtime error.  
- **HTTP xxx** → backend returned error code (e.g., HTTP 500).  

The table shows error counts per (users × mode) combination, making it easy to compare how error types scale across load levels.

**Goal**:  
- Detect whether failures are correlated with concurrency (e.g., more errors at 200+ users).  
- Compare modes to see if one degrades faster under load.  
- Spot backend instability vs. client/network issues.  

We will also include a few sample error messages for context after the table.

In [10]:
# --- Error Diagnostics Summary ---

# Group by users, mode, and status_code to count errors
error_summary = (
    progressive_results_df[~progressive_results_df["success"]]
    .groupby(["users", "mode", "status_code"])
    .size()
    .reset_index(name="count")
)

if error_summary.empty:
    print("\n## 📉 Error Diagnostics")
    print("No errors captured in any run.")
else:
    print("\n## 📉 Error Diagnostics Summary (per users × mode × status code)")
    print(error_summary.to_markdown(index=False))

    # Show a few sample error messages for context
    print("\n### 🔎 Sample Error Messages")
    sample_errors = progressive_results_df[~progressive_results_df["success"]].head(5)
    for _, row in sample_errors.iterrows():
        print(f"- [Users={row['users']}, Mode={row['mode']}, Code={row['status_code']}] {row['error']}")



## 📉 Error Diagnostics Summary (per users × mode × status code)
|   users | mode        | status_code   |   count |
|--------:|:------------|:--------------|--------:|
|     100 | insight     | 502           |      75 |
|     100 | performance | 502           |      36 |
|     100 | performance | TIMEOUT       |       1 |
|     200 | insight     | 502           |     194 |
|     200 | performance | 502           |     175 |
|     300 | insight     | 502           |     295 |
|     300 | performance | 502           |     264 |

### 🔎 Sample Error Messages
- [Users=100, Mode=insight, Code=502] HTTP 502: {"success":false,"error":"LLM response must be a string","reason":"internal_error"}
- [Users=100, Mode=performance, Code=502] HTTP 502: {"success":false,"error":"Inference skipped","reason":"missing_signals","diagnostics":{"observations":{},"predictions":{},"status":"degraded","message":"Observation selection failed because the upstre
- [Users=100, Mode=insight, Code=502] HTTP 502: {"suc

#### Output Consistency Anomalies
In addition to explicit errors, we detect **anomalies in output structure**:
- **Insight mode**: 
  - Empty observations → unexpected anomaly.  
  - Missing descriptions → unexpected anomaly.  
- **Performance mode**:  
  - Empty observations → unexpected anomaly.  
  - Missing descriptions → acceptable (by design).  

The anomaly summary table shows counts of these inconsistencies per (users × mode).  
These are crucial for identifying **backend race conditions, caching effects, or data loss under load**,  
even when requests technically succeed (status 200).

In [11]:
# --- Anomaly Detection for Output Consistency ---

# This analysis complements the Error Diagnostics by checking for logical/content anomalies.
# It ensures that outputs conform to expectations per mode (insight vs performance).
def detect_output_anomalies(progressive_results_df: pd.DataFrame) -> pd.DataFrame:
    anomalies = []

    # --- Check Insight mode anomalies ---
    insight_groups = progressive_results_df[progressive_results_df["mode"] == "insight"]
    for users, group in insight_groups.groupby("users"):
        # Case 1: Empty observations {}
        empty_obs_count = group["response_text"].str.contains('"observations":{}').sum()
        if empty_obs_count > 0:
            anomalies.append({
                "users": int(users),
                "mode": "insight",
                "issue": "Empty observations detected (unexpected)",
                "count": int(empty_obs_count),
                "total": len(group)
            })

        # Case 2: Observations exist but missing descriptions ""
        missing_desc_count = group["response_text"].str.contains('"description":""').sum()
        if missing_desc_count > 0:
            anomalies.append({
                "users": int(users),
                "mode": "insight",
                "issue": "Observations missing descriptions (unexpected)",
                "count": int(missing_desc_count),
                "total": len(group)
            })
        
    # --- Check Performance mode anomalies ---
    perf_groups = progressive_results_df[progressive_results_df["mode"] == "performance"]
    for users, group in perf_groups.groupby("users"):
        # Case: Empty observations {} 
        empty_obs_count = group["response_text"].str.contains('"observations":{}').sum()
        if empty_obs_count > 0:
            anomalies.append({
                "users": int(users),
                "mode": "performance",
                "issue": "Empty observations detected (unexpected)",
                "count": int(empty_obs_count),
                "total": len(group)
            })

    # Convert anomaly list to DataFrame
    anomalies_df = pd.DataFrame(anomalies)
    return anomalies_df


# --- Run anomaly detection ---
anomalies_df = detect_output_anomalies(progressive_results_df)

print("## 🔎 Output Anomalies Summary (per users × mode × issue)")
if anomalies_df.empty:
    print("No output anomalies detected ✅")
else:
    print(anomalies_df.to_markdown(index=False))


## 🔎 Output Anomalies Summary (per users × mode × issue)
No output anomalies detected ✅


### Visualizations

This section generates plots to visualize stress test behavior.  
Visualizations are produced at both the **concurrency level** and the **mode level**.  

1. **Throughput Scaling Curve (run-level only)**  
   - Shows the **average sustained throughput (requests/sec)** achieved at each concurrency level.  
   - Both modes are aggregated to capture total system capacity under load.  
   - Ideally, throughput should rise as concurrency increases. If it plateaus or drops, the system has reached a **scaling limit or bottleneck**.  

2. **Latency Scaling Curve (mean vs users, split by mode)**  
   - Plots the **average response time (mean latency)** for insight and performance modes separately.  
   - A **flat line** means the system absorbs more load without slowing down.  
   - An **upward slope** means requests are taking longer, showing resource stress.  
   - A **downward slope** can indicate batching/optimizations, but may also signal **anomalies** (e.g., skipped work or inconsistent outputs).  

3. **Latency Histograms (per concurrency × mode)**  
   - Separate histograms for insight and performance requests at each concurrency level.  
   - Highlights the distribution of response times, showing whether variance and tail latency (slowest requests) grow as load increases.  

**Goal:** Together, these plots provide a holistic view of scaling:  
- **Throughput** shows how much work the system can sustain.  
- **Latency curves** show how response times evolve under stress.  
- **Histograms** show the distributional effects hidden in averages.  
This allows us to compare scaling behavior across modes and detect bottlenecks or anomalies.  

In [12]:
# --- Latency Histograms (separate per users × mode) ---
for (users, mode), group in progressive_results_df.groupby(["users", "mode"]):
    successful = group[group["success"]]
    if successful.empty:
        continue
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(
        successful["latency_ms"],
        bins=30,
        color="#1f77b4" if mode == "insight" else "#ff7f0e",
        edgecolor="black",
        alpha=0.8
    )
    ax.set_title(f"Latency Distribution ({users} Users, {mode})")
    ax.set_xlabel("Latency (ms)")
    ax.set_ylabel("Request Count")
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))  # integer y-axis
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
    fig.tight_layout()
    fig.savefig(combined_dir / f"latency_hist_{users}_users_{mode}.png", dpi=200)
    plt.close(fig)

# --- Throughput Scaling Curve (run-level only) ---
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(
    progressive_runs_df["users"],
    progressive_runs_df["throughput_rps"],
    marker="o",
    linewidth=2,
    label="Throughput"
)
ax.set_title("Throughput Scaling Curve (Run-Level)")
ax.set_xlabel("Concurrent Users")
ax.set_ylabel("Requests per Second")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.grid(True, alpha=0.3)
ax.legend(frameon=True, facecolor="white", edgecolor="black")
fig.tight_layout()
fig.savefig(combined_dir / "progressive_throughput.png", dpi=150)
plt.close(fig)

# --- Latency Scaling (mean vs users, split by mode) ---
fig, ax = plt.subplots(figsize=(10, 6))
for mode, group in latency_summary_df.groupby("mode"):
    ax.plot(group["users"], group["mean_ms"], marker="o", label=f"{mode} - Mean")
ax.set_title("Latency Scaling with Concurrency (per Mode)")
ax.set_xlabel("Concurrent Users")
ax.set_ylabel("Latency (ms)")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(frameon=True, facecolor="white", edgecolor="black")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(combined_dir / "latency_scaling_modes.png", dpi=150)
plt.close(fig)


## 8. Scaling Recommendations with OpenAI

This section uses OpenAI to generate **scaling recommendations** for the Inhibitor API,  
based on the results of the **progressive stress test** (20 → 50 → 100 → 200 → 300 users).

Unlike the diagnostic run, which validates correctness under a single configuration,  
the progressive run highlights **system behavior under increasing concurrency**.  
It shows where throughput begins to plateau, where latency spikes, and where errors emerge.

The analysis is **tailored to our deployment on Cloudflare Workers**, and considers:

- Horizontal scaling across Workers and geographic regions  
- Use of Durable Objects, KV, or Queues for coordination under load  
- Connection pooling and request distribution at the edge  
- Edge caching or batching strategies to reduce duplicate evaluation work  
- Worker CPU/memory execution limits and timeouts  
- Timeout and retry tuning to balance strict SLAs vs natural queuing  

In addition, this step explicitly checks for **output consistency anomalies** across all modes:  
- In **insight mode**, anomalies where the same scenario produced both **detailed** and **empty** outputs under load.  
- In **performance mode**, unexpected collapses to empty observations (not just missing descriptions).  
These behaviors are strong signals of **backend race conditions, queuing artifacts, or Cloudflare Worker caching effects**.  
Recommendations will include mitigations such as cache-busting, per-request IDs, and stronger determinism.

The recommendations are reported in three focused sections:  

1. **Scaling Recommendations** – infra-level guidance tied to throughput/latency/error data.  
2. **Output Consistency Anomalies** – cross-mode checks for correctness under stress.  
3. **Insight Mode Consistency** – zoomed-in review of insight-mode reliability under load.  
4. **Immediate Risks** – what could break if load increases further without intervention.  

These recommendations are **holistic** — grounded in the full progression of load levels,  
so we can identify **breaking points** and understand how to scale effectively before production rollout.  


In [13]:
def evaluate_progressive_stress_with_openai(
    progressive_summary_df: pd.DataFrame,
    latency_summary_df: pd.DataFrame,
    progressive_results_df: pd.DataFrame,
    anomalies_df:pd.DataFrame
) -> str:
    """
    Use OpenAI to generate scaling recommendations based on progressive load test data.
    Tailored for Inhibitor API running on Cloudflare Workers.
    Includes anomaly detection for inconsistent outputs and latency slope analysis.
    """

    # Error breakdown grouped by concurrency level
    error_df = progressive_results_df[~progressive_results_df["success"]].copy()
    error_summary = (
        error_df.groupby(["users", "mode", "status_code"])
        .size()
        .reset_index(name="count")
        .to_dict(orient="records")
    )

    # Convert DataFrames to dicts for prompt readability
    progressive_summary = progressive_summary_df.to_dict(orient="records")
    latency_summary = latency_summary_df.to_dict(orient="records")
    anomalies_summary = anomalies_df.to_dict(orient="records") if anomalies_df is not None else []

    # Build detailed prompt
    prompt = f"""
You are analyzing a **progressive stress test** for the Inhibitor API,
which is deployed on **Cloudflare Workers**.

**Progressive Run Summary**
{json.dumps(progressive_summary, indent=2)}

**Latency Statistics (ms)**
{json.dumps(latency_summary, indent=2)}

**Error Breakdown**
{json.dumps(error_summary, indent=2)}

**Output Anomalies (consistency issues)**
{json.dumps(anomalies_summary, indent=2)}

**Task for you:**
1. Identify **where scaling starts to break down** (latency spikes, errors, throughput plateau).
2. Provide **scaling recommendations tailored to Cloudflare Workers**, including:
   - Horizontal scaling across Workers
   - Durable Objects, KV, or Queues for coordination
   - Edge caching or batching
   - Connection pooling considerations
   - Timeout/retry tuning
   - Worker CPU/memory execution timeouts
3. Highlight **immediate risks** if load continues to increase (timeouts, queue backpressure, Worker execution limits).
4. Pay special attention to **output consistency anomalies across all modes**.
   - In *insight* mode, empty observations or missing descriptions are unexpected.
   - In *performance* mode, empty observations are unexpected, but missing descriptions may be acceptable.
5. Explicitly check for **latency scaling anomalies**: 
   - A **flat latency curve** means the system handles added load without slowing down.
   - An **upward slope** means response times are degrading as load increases.
   - A **downward slope** under higher concurrency is a **red flag**:
     it often means outputs are collapsing to empty or incomplete responses,
     which are returned faster but indicate correctness failures, not performance gains.
6. Tie each recommendation directly to the observed data above.
Do not provide generic best practices.

Return your answer in these sections:

### Scaling Recommendations
(bullets, grounded in run data; tailored to Cloudflare Workers)

### Output Consistency Anomalies
(bullets, highlight anomalies across all modes; note which are expected vs unexpected, and propose general fixes)

### Insight Mode Consistency
(bullets, deeper dive into why empty vs detailed outputs occurred in insight mode, and recommend mitigations like cache-busting, request IDs, and determinism)

### Immediate Risks
(bullets, grounded in run data; describe what could fail if load continues without scaling)
"""


    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a senior performance engineer specializing in Cloudflare Workers."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content


In [14]:
# Generate recommendations using OpenAI
openai_recommendations = evaluate_progressive_stress_with_openai(
    progressive_summary_df, latency_summary_df, progressive_results_df, anomalies_df
)
print(openai_recommendations)

### Scaling Recommendations

- **Horizontal Scaling Across Workers**: The throughput begins to plateau and errors increase significantly at 100 users, indicating that the current deployment may be reaching its limits. Consider deploying additional Workers to distribute the load more effectively across multiple instances.

- **Durable Objects for Coordination**: The error rate at higher user counts suggests potential state management issues. Implement Durable Objects to manage state and coordinate between Workers, reducing the likelihood of 502 errors due to state inconsistencies.

- **Edge Caching**: Implement edge caching for frequently accessed data to reduce the load on Workers and improve response times. This can help mitigate latency spikes observed at higher user counts.

- **Batching Requests**: Consider batching requests where possible to reduce the number of individual requests hitting the Workers, which can help manage throughput and reduce latency.

- **Timeout/Retry Tuning*

## 9. Save Markdown Report

Persist the raw data, summary metrics, plots, and OpenAI recommendations into a Markdown for further analysis and reporting.


In [15]:
# --- Prepare report file path ---
report_path = combined_dir / f"inhibitor_progressive_stress_report.md"

# --- Methodology ---
methodology_block = """# Inhibitor Progressive Stress Test Report

## Official Citation for the Inhibitor Progressive Stress Testing (IPST)

```bibtex
@software{inhibitorlab2025,
  title     = {Inhibitor Progressive Stress Testing (IPST): A Framework for Evaluating Concurrency Limits and Performance Scalability of the Inhibitor API},
  author    = {appliedAIstudio and contributors},
  year      = {2025},
  publisher = {Inhibitor-Lab Project},
  note      = {Stress Test ID: IPST-2025-V1.11}
}
```

## Methodology

This stress testing suite complements the **Inhibitor Evaluation Benchmark (IEB)** by focusing not on output quality,
but on **scalability, latency under load, and system resilience**.

### Diagnostic Run (5 × 2 users/requests)
- A small-scale dry run to ensure configuration and error handling work as expected.  
- Captures sample responses and baseline latency.  

### Progressive Load Tests & Scenario Assignment
- Concurrency is increased stepwise (20, 50, 100, 200, 300).  
- Each step measures throughput, latency distribution, and errors.  
- Traffic is **mixed-mode** (alternating `"insight"` and `"performance"` requests) to better reflect real-world usage.  

- A fixed set of 6 benchmark scenarios is reused throughout all progressive runs.  
- Each simulated user issues **2 requests**:  
  - One in *insight* mode.  
  - One in *performance* mode.  
- Scenarios are assigned in round-robin order across users.  

**Why this matters:**  
- Ensures both modes are tested under the **same scenarios**, controlling for scenario variability.  
- Produces directly comparable results between modes.  
- Improves statistical coverage without inflating total request counts.  

### Metrics Captured
- **Latency statistics**: min, mean, median, max (per users × mode).  
- **Throughput**: average sustained requests/sec per concurrency level.  
- **Errors**: categorized into TIMEOUT, EXCEPTION, or API_ERROR (per users × mode). 
- **Output consistency anomalies**: empty observations, missing descriptions, or unexpected divergence between modes.  
- **Visualization outputs**: latency histograms (per users × mode), throughput scaling, latency scaling.  

This methodology enables us to track how throughput and latency evolve as concurrency increases,  
diagnose potential bottlenecks (network, Cloudflare Worker limits, or concurrency issues),  
and flag **output consistency anomalies** that may indicate race conditions or caching artifacts under load.

"""

# --- Config section ---
config_block = f"""## Test Configuration
- API Version: {INHIBITOR_API_VERSION}
- Concurrency Levels Tested: {', '.join(map(str, sorted(progressive_summary_df['users'].unique())))}
- Modes Tested: {', '.join(progressive_summary_df['mode'].unique())} (alternating per user)
- Requests per User: 2 (one insight request + one performance request)
- Timeout (s): {TIMEOUT}
"""

# --- Scenario Set ---
scenario_set_block = "## Scenario Set\n```json\n[\n"

for i, scenario in enumerate(scenarios):
    scenario_entry = {
        "scenario_id": scenario.get("scenario_id", f"SCENARIO_{i+1:03d}"),
        "scenario_title": scenario.get("title", ""),
        "thought_chain": scenario.get("thought_chain", [])
    }
    scenario_json = json.dumps(scenario_entry, indent=2)
    scenario_set_block += scenario_json
    if i < len(scenarios) - 1:
        scenario_set_block += ",\n"
    else:
        scenario_set_block += "\n"

scenario_set_block += "]\n```"

# --- Latency Summary ---
latency_summary_markdown = latency_summary_df.to_markdown(index=False) if not latency_summary_df.empty else "No successful requests to summarize."
latency_block = f"""## Latency Summary

Latency statistics across different concurrency levels and modes (only successful responses included):

{latency_summary_markdown}
"""

# --- Throughput Summary ---
throughput_summary_markdown = progressive_runs_df[[
    "users", "total_requests", "duration_sec", "throughput_rps"
]].to_markdown(index=False)

throughput_block = f"""## Throughput Summary

This section reports the **average sustained requests per second (req/sec)** at each tested concurrency level (20 → 300 users).  
Throughput measures the system's processing capacity under load.  

Throughput is calculated as:

`throughput (req/sec) = total requests completed / run duration (seconds)`

**Interpretation:**  
Throughput should scale upward as user concurrency increases.  
If throughput **stops increasing**, **plateaus**, or **drops**, the system has likely reached a bottleneck.  

{throughput_summary_markdown}
"""

# --- Visualizations ---
plots_block = f"""## Visualizations

The following plots illustrate how the system behaves under progressive load:

- **Throughput scaling curve** shows the **average sustained throughput (requests/sec)** achieved at each concurrency level.
- **Latency scaling curves** track the *average (mean) latency* at each concurrency level, split by mode. 
- **Latency histograms** show the distribution of response times at each concurrency level × mode.   

![Progressive Throughput](progressive_throughput.png)

*Throughput scaling curve (users → requests/sec). 
Ideally, throughput should rise as concurrent users increase. 
If throughput plateaus or drops as concurrency increases, the system has hit a scaling limit.*

![Latency Scaling by Mode](latency_scaling_modes.png)

*Latency scaling curve (users → latency in ms).*  
- *A **flat line** means the system handles additional load without slowing down.*  
- *An **upward slope** indicates increasing response delays under load (stress on system resources).*  
- *A **downward slope** may occur if requests are being processed faster or batching is occurring, but can also hint at anomalies (e.g., skipped work, inconsistent outputs).*

"""

# Add latency histograms per concurrency level × mode
for (users, mode), group in progressive_results_df.groupby(["users", "mode"]):
    if users <= 300:
        plots_block += f"""
### Latency Histogram - {users} Users ({mode.capitalize()} Mode)

Distribution of response times (ms) when running with **{users} concurrent users** in **{mode} mode**.  
This helps highlight latency shifts under load for each mode.

![Latency Histogram - {users} Users {mode.capitalize()}](latency_hist_{users}_users_{mode}.png)
"""

# --- Error Diagnostics & Output Consistency ---
error_block = "## Error Diagnostics & Output Consistency Anomalies\n\n"

# Section explanation
error_block += (
    "This section reports both **non-successful responses** and **output anomalies** observed "
    "during progressive load tests:\n\n"
    "- **Error Diagnostics**: Counts of failed requests (timeouts, exceptions, API errors), "
    "broken down by concurrency level × mode.\n"
    "- **Output Consistency Anomalies**: Cases where outputs deviated from expectations "
    "(e.g., empty observations in *insight* mode, or empty observations in *performance* mode "
    "where at least a value/index should exist).\n\n"
    "**Goal:** Detect not just outright failures, but also silent correctness issues that "
    "emerge under higher load.\n\n"
    "### Error Diagnostics Summary\n"
)

any_errors = False

for (users, mode), group in progressive_results_df.groupby(["users", "mode"]):
    subset = group[~group["success"]]
    if subset.empty:
        continue  # skip if no errors
    any_errors = True

    # Count errors by status code
    error_summary = (
        subset.groupby("status_code")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    error_block += f"### {users} Users – {mode.capitalize()} Mode\n"
    error_block += error_summary.to_markdown(index=False) + "\n\n"

if not any_errors:
    error_block += "No errors were captured in any run.\n\n"

# Output consistency anomalies
if anomalies_df is not None and not anomalies_df.empty:
    error_block += "### Output Consistency Anomalies\n"
    error_block += (
        "This subsection tracks **silent correctness issues** — cases where outputs did not match expectations.\n\n"
        "**Mode-specific expectations:**\n"
        "- **Insight mode**:\n"
        "  - Empty observations → unexpected anomaly.\n"
        "  - Missing descriptions → unexpected anomaly (this happens when the API flags an observation but does not provide the explanation of *why* it was flagged).\n"
        "- **Performance mode**:\n"
        "  - Empty observations → unexpected anomaly.\n"
        "  - Missing descriptions → acceptable (by design, since performance mode prioritizes speed over explanation detail).\n\n"
        "The table below summarizes anomaly counts per (users × mode).\n\n"
    )
    error_block += anomalies_df.to_markdown(index=False) + "\n\n"
else:
    error_block += "### Output Consistency Anomalies\nNo anomalies detected.\n\n"

    
# --- Recommendations (if available) ---
recommendations_block = "## OpenAI Scaling Recommendations\n\n"
try:
    recommendations_block += openai_recommendations
except NameError:
    recommendations_block += "_(Not generated)_"

# --- Combine all sections ---
report_content = "\n".join([
    methodology_block,
    config_block,
    scenario_set_block,
    latency_block,
    throughput_block,
    plots_block,
    error_block,
    recommendations_block,
])

# --- Save report ---
report_path.write_text(report_content, encoding="utf-8")
print("📄 Markdown report saved:", report_path)

📄 Markdown report saved: results/v1.21.0/inhibitor_progressive_stress_report.md



Stress test artifacts, plots, and the markdown report have been saved. Review the results directory for audit-ready outputs.
